# NB_00 — SOURCE_03 Extraction

**Source:** *Electroplating Deposition of Bismuth Absorbers for X-ray Superconducting Transition Edge Sensors*

This notebook completes the `SOURCE_03` scaffold through the registered reusable extractor, validates the completed source record, writes the canonical YAML, and creates a downloadable export package.

Expected repository files:

```text
engineering_navigator/absorber_manufacturing/source_records/
    SOURCE_03_electroplating_process.scaffold.yaml

tools/source_extractors/
    source_03.py
    registry.py
```


## 1. Configuration and repository discovery

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE = None

SOURCE_ID = "SOURCE_03"
SCAFFOLD_FILENAME = "SOURCE_03_electroplating_process.scaffold.yaml"
SOURCE_FILENAME = "SOURCE_03_electroplating_process.yaml"
PAPER_TITLE = (
    "Electroplating Deposition of Bismuth Absorbers for X-ray "
    "Superconducting Transition Edge Sensors"
)


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])

    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")

        if target.exists():
            if (target / "engineering_navigator").is_dir():
                return target
            raise FileExistsError(
                f"{target} exists but does not look like sensors-becker."
            )

        subprocess.run(["git", "clone", REPOSITORY_URL, str(target)], check=True)

        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly."
    )


ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SOURCE_DIR = (
    ROOT
    / "engineering_navigator"
    / "absorber_manufacturing"
    / "source_records"
)
SCAFFOLD_PATH = SOURCE_DIR / SCAFFOLD_FILENAME
SOURCE_PATH = SOURCE_DIR / SOURCE_FILENAME

EXPORT_DIR = ROOT / "exports" / SOURCE_ID
EXPORT_ZIP = ROOT / "exports" / f"{SOURCE_ID}_export.zip"

print(f"Repository : {ROOT}")
print(f"Scaffold   : {SCAFFOLD_PATH.relative_to(ROOT)}")
print(f"Output     : {SOURCE_PATH.relative_to(ROOT)}")


## 2. Verify extractor registration

In [ ]:
from tools.source_extractors.registry import EXTRACTORS, extract_source

available = sorted(EXTRACTORS)
print("Registered extractors:", available)

if SOURCE_ID not in EXTRACTORS:
    raise ValueError(
        f"{SOURCE_ID} is not registered. Available: {available}"
    )

print("Registry validation: PASS")


## 3. Load and validate scaffold

In [ ]:
if not SCAFFOLD_PATH.exists():
    raise FileNotFoundError(
        f"Missing scaffold: {SCAFFOLD_PATH}\n"
        "Add SOURCE_03_electroplating_process.scaffold.yaml to the repo."
    )

scaffold = yaml.safe_load(SCAFFOLD_PATH.read_text(encoding="utf-8"))

if not isinstance(scaffold, dict):
    raise TypeError("SOURCE_03 scaffold must contain one top-level mapping.")

if scaffold.get("source_id") != SOURCE_ID:
    raise ValueError(
        f"Expected source_id {SOURCE_ID!r}; found {scaffold.get('source_id')!r}"
    )

if scaffold.get("title") != PAPER_TITLE:
    print("WARNING: scaffold title differs from notebook PAPER_TITLE.")

print("Loaded:", scaffold.get("title"))
print("Status:", scaffold.get("extraction_status"))


## 4. Run SOURCE_03 extractor

In [ ]:
completed_record = extract_source(SOURCE_ID, scaffold)

print(f"Materials                 : {len(completed_record.get('materials', []))}")
print(f"Fabrication methods       : {len(completed_record.get('fabrication_methods', []))}")
print(f"Design variables          : {len(completed_record.get('design_variables', []))}")
print(f"Reported values           : {len(completed_record.get('reported_values', []))}")
print(f"Measured outcomes         : {len(completed_record.get('measured_outcomes', []))}")
print(f"Engineering relationships : {len(completed_record.get('engineering_relationships', []))}")
print(f"Engineering constraints   : {len(completed_record.get('engineering_constraints', []))}")
print(f"Reported process points   : {len(completed_record.get('reported_process_points', []))}")


## 5. Validate completed source record

In [ ]:
required_nonempty = [
    "authors",
    "materials",
    "fabrication_methods",
    "design_variables",
    "reported_values",
    "measured_outcomes",
    "engineering_relationships",
    "engineering_constraints",
    "future_questions",
]

errors = []

if completed_record.get("source_id") != SOURCE_ID:
    errors.append("source_id changed during extraction")

if completed_record.get("record_status") != "evidence_extracted":
    errors.append("record_status must be evidence_extracted")

if completed_record.get("extraction_status") != "complete_for_source_record_v1":
    errors.append("extraction_status must be complete_for_source_record_v1")

for key in required_nonempty:
    value = completed_record.get(key)
    if not isinstance(value, list) or len(value) == 0:
        errors.append(f"{key} must be a non-empty list")

process_points = completed_record.get("reported_process_points", [])
if not isinstance(process_points, list) or len(process_points) == 0:
    errors.append("reported_process_points must contain at least one point")

for i, point in enumerate(process_points):
    if not isinstance(point, dict):
        errors.append(f"reported_process_points[{i}] must be a mapping")
        continue
    if point.get("source") != SOURCE_ID:
        errors.append(
            f"reported_process_points[{i}].source must equal {SOURCE_ID}"
        )

if errors:
    print("Validation: FAIL")
    for error in errors:
        print("-", error)
    raise ValueError("SOURCE_03 validation failed.")

print("Validation: PASS")


## 6. Inspect process evidence

In [ ]:
print("Reported process points:")
print(
    yaml.safe_dump(
        completed_record.get("reported_process_points", []),
        sort_keys=False,
        allow_unicode=True,
    )
)

print("Unreported variables:")
for item in completed_record.get("unreported_variables", []):
    print("-", item)


## 7. Write canonical SOURCE_03 YAML

In [ ]:
SOURCE_PATH.write_text(
    yaml.safe_dump(
        completed_record,
        sort_keys=False,
        allow_unicode=True,
        width=110,
    ),
    encoding="utf-8",
)

print(f"Wrote: {SOURCE_PATH.relative_to(ROOT)}")
print(f"Bytes: {SOURCE_PATH.stat().st_size:,}")


## 8. Reload written YAML

In [ ]:
reloaded = yaml.safe_load(SOURCE_PATH.read_text(encoding="utf-8"))

if reloaded != completed_record:
    raise ValueError("Reloaded YAML differs from completed_record.")

print("Round-trip YAML validation: PASS")
print("Loaded:", reloaded["title"])
print("Status:", reloaded["extraction_status"])


## 9. Create export package and download

In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

export_files = [SOURCE_PATH]

for path in export_files:
    shutil.copy2(path, EXPORT_DIR / path.name)

manifest = {
    "source_id": SOURCE_ID,
    "title": completed_record["title"],
    "extraction_status": completed_record["extraction_status"],
    "files": [path.name for path in export_files],
    "counts": {
        "materials": len(completed_record.get("materials", [])),
        "fabrication_methods": len(completed_record.get("fabrication_methods", [])),
        "design_variables": len(completed_record.get("design_variables", [])),
        "reported_values": len(completed_record.get("reported_values", [])),
        "engineering_relationships": len(
            completed_record.get("engineering_relationships", [])
        ),
        "reported_process_points": len(
            completed_record.get("reported_process_points", [])
        ),
    },
}

manifest_path = EXPORT_DIR / "manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(EXPORT_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Export package: {EXPORT_ZIP}")
print(f"Size: {EXPORT_ZIP.stat().st_size:,} bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## Handoff

After this notebook passes:

```text
SOURCE_03 scaffold
        ↓
source_03.py
        ↓
registry
        ↓
NB_00_SOURCE_03_EXTRACTION
        ↓
SOURCE_03_electroplating_process.yaml
```

The completed record can then feed the Engineering Object builder and the electroplated-Bi process-window pipeline.
